In [ ]:
#Variables
import numpy as np
# MASSES
m_s = 1600 #kg  # Vehicle Cab Mass
m_uf = 40 #kg   tire+rim+brakes
m_ur = 40 #kg   tire+rim+brakes
# CONSTANTS
# RIGHT SIDE
k_sf1 = 60000 # N/m   Front Wheel          #60000 to 150000+ for race cars
c_sf1 = 10000 # Ns/m   Front Wheel
k_sr1 = 60000 # N/m   Rear Wheel
c_sr1 = 10000 # Ns/m   Rear Wheel
k_tf1 = 200000 # N/m   tire stiffness
k_tr1 = 200000 # N/m
# LEFT SIDE
k_sf2 = 60000 # N/m   Front Wheel          #60000 to 150000+ for race cars
c_sf2 = 10000 # Ns/m   Front Wheel
k_sr2 = 60000 # N/m   Rear Wheel
c_sr2 = 10000 # Ns/m   Rear Wheel
k_tf2 = 200000 # N/m   tire stiffness
k_tr2 = 200000 # N/m

# POSITION AND VELOCITY VARIABLES
z_s = 0.0  #Car center of mass position/displacement
# RIGHT SIDE
z_uf_RT = 0.0      #Front tire position/displacement
z_ur_RT = 0.0      #Rear tire position
z1_s_RT = 0.0      #Car center of mass velocity
z1_uf_RT = 0.0     #Front tire velocity
z1_ur_RT = 0.0     #Rear tire velocity
#LEFT SIDE
z_uf_LT = 0.0      #Front tire position/displacement
z_ur_LT = 0.0      #Rear tire position
z1_s_LT = 0.0      #Car center of mass velocity
z1_uf_LT = 0.0     #Front tire velocity
z1_ur_LT = 0.0     #Rear tire velocity


t = 0.0 # seconds
# WHEELBASE DIMENSIONS
a = 1.2 # meters  distance from CG to front axle
b = 1.8 # meters  distance from CG to rear axle
c = 0.9 # meters  distance from CG to RIGHT wheels
d = 0.9 # meters  distance from CG to LEFT wheels
h = 0.16 # meters        Ride Height
r_tire = 0.2286 #meters  Tire Radius

# PITCH AND ROLL VARIABLES
theta = 0.0 #radians     Car Pitch
theta1 = 0.0
ro = 0.0  #radians       Car Roll
ro1 = 0.0

g = 9.81 # m/s^2
z_r = 0.0
vel = 20 #m/s
delay = (a+b)/vel

# SUSPENSION BOUNDS
max_travel = 0.1524      #MAX suspension travel
min_travel = 0.0508      #MIN suspension travel
k_b = 100000 #N/m       #Suspension Bumper "spring constant"

In [ ]:
#Plotting Functions
def simple_plot(x_value, y_value, xlabel, ylabel, title):
  import matplotlib.pyplot as plt
  plt.figure()
  plt.plot(x_value, y_value)
  plt.xlabel(f'{xlabel}')
  plt.ylabel(f'{ylabel}')
  plt.title(f'{title}' )
  plt.grid
  plt.show()


def plot_overlay(x_value, y1_value, y2_value, xlabel, ylabel, title, line1, line2):
  import matplotlib.pyplot as plt
  plt.figure()
  plt.plot(x_value, y1_value, label=line1)
  plt.plot(x_value, y2_value, label=line2, linestyle='--')
  plt.xlabel(f'{xlabel}')
  plt.ylabel(f'{ylabel}')
  plt.title(f'{title}')
  plt.grid()
  plt.legend()
  plt.show()

def tripleplot_overlay(x_value, y1_value, y2_value, y3_value, xlabel, ylabel, title, line1, line2, line3):
  import matplotlib.pyplot as plt
  plt.figure()
  plt.plot(x_value, y1_value, label=line1)
  plt.plot(x_value, y2_value, label=line2, linestyle='--')
  plt.plot(x_value, y3_value, label=line3, linestyle='-.')
  plt.xlabel(f'{xlabel}')
  plt.ylabel(f'{ylabel}')
  plt.title(f'{title}')
  plt.grid()
  plt.legend()
  plt.show()

In [ ]:
#Random Road input
import random
def rand_roadinput(t):
  if t < 1.000 or t > 1.100 and t < 2.000 or t > 2.100:
    #z_r = 0.0
    z_r = random.uniform(-0.005, 0.005)
  if t >= 1.000 and t <= 1.100:
    z_r = random.uniform(0.00, 0.11)
  if t >= 2.000 and t <= 2.100:
    z_r = random.uniform(-0.01, 0.00)
  return z_r

In [ ]:
#Car acceleration  a = F/m
def FullCar_Accel(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, delay, t):
  #Road Input
  z_rf = rand_roadinput(t)
  z_rr = rand_roadinput(t-delay)
  #Deflections
  dsf_RT = (z_s + (a*theta)) - z_uf    # a*theta is arc length
  dsr_RT = (z_s - (b*theta)) - z_ur
  dsf_LT =
  dsr_RT =
  #Deflection Velocity
  d1sf = (z1_s + a*theta1) - z1_uf
  d1sr = (z1_s - b*theta1) - z1_ur
  d1sf_LT =
  d1sr_RT =
  #Forces and Moments
  F_sf = (k_sf*dsf) + (c_sf*d1sf)
  F_sr = (k_sr*dsr) + (c_sr*d1sr)

  Ma = F_sf * a
  Mb = F_sr * b

  if min_travel > dsf > max_travel:                #Bounding the suspension
    F_sf += k_b*(dsf - max_travel)
  if dsr > max_travel:
    F_sr += k_b*(dsr - max_travel)

  #Inertia
  Iyy = m_s*((a+b)**2)*0.2  #0.2 is scaling factor

  #Accelerations
  a_s = ((-k_sf*(z_s - z_uf))-(k_sr*(z_s - z_ur))-(c_sf*(z1_s - z1_uf))-(c_sr*(z1_s - z1_ur))) / m_s
  a_uf = ((k_sf*(z_s - z_uf)) + (c_sf*(z1_s - z1_uf)) - (k_tf*(z_uf - z_rf))) / m_uf
  a_ur = ((k_sr*(z_s - z_ur)) + (c_sr*(z1_s - z1_ur)) - (k_tr*(z_ur - z_rr))) / m_ur
  a_theta = (-a*(F_sf) + b*(F_sr)) / (Iyy)  #mass cancels out
  return a_s, a_uf, a_ur, a_theta, z_rf, z_rr

a_s, a_uf, a_ur, a_theta, z_rf, z_rr = HalfCar_Accel(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, delay, t)

In [ ]:
#RK4 Step
def RK4_step(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, t):

  def Q(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, t):
    a_s, a_uf, a_ur, a_theta, z_rf, z_rr = FullCar_Accel(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, delay, t)
    return np.array([z1_s, a_s, z1_uf, a_uf, z1_ur, a_ur, theta1, a_theta])

  X = np.array([z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1])

  k1 = Q(*X, t)
  k2 = Q(*(X + (0.5*k1*dt)), (t + 0.5*dt))
  k3 = Q(*(X + (0.5*k2*dt)), (t + 0.5*dt))
  k4 = Q(*(X + (k3*dt)), (t + dt))

  X_next = X + ((k1 + 2*k2 + 2*k3 + k4)/6) * dt
  return X_next

In [ ]:
#RK4 Solver
t = 0.0
t_end_rk4 = 3.0 #sec
dt = 0.01

Time = []
Zs = []
Zuf = []
Zur = []
Z1s = []
Z1uf = []
Z1ur = []
Pitch = []
Pitch_Vel = []
Zrf = []
Zrr = []
A_s = []
A_ur = []

while t < t_end_rk4:
  a_s, a_uf, a_ur, a_theta, z_rf, z_rr = HalfCar_Accel(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, delay, t)
  z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1 = RK4_step(z_s, z1_s, z_uf, z1_uf, z_ur, z1_ur, theta, theta1, t)
  #ENG = (0.5*m_s*z1_s**2)+(0.5*k_sf*(z))
  t += dt

  Time.append(t)
  Zs.append(z_s)
  Zuf.append(z_uf)
  Zur.append(z_ur)
  Z1s.append(z1_s)
  Z1uf.append(z1_uf)
  Z1ur.append(z1_ur)
  Pitch.append(theta)
  Pitch_Vel.append(theta1)
  Zrf.append(z_rf)
  Zrr.append(z_rr)
  A_s.append(a_s)
  A_ur.append(a_ur)

In [ ]:
import csv
filename = f"HalfCar_Data1.csv"
with open(filename, "w", newline="") as f:
    writer = csv.writer(f)

     #Header
    writer.writerow([
        "Time_s",
        "SprungMassPos_m",
        "F_WheelPos_m",
        "R_WheelPos_m",
        "SprungVel",
        "F_WheelVel",
        "R_WheelVel",
        "Pitch",
        "PitchVel",
        "RoadInputFront_m",
        "RoadInputRear_m"
      ])

    #Time History
    for t, Z_s, Z_uf, Z_ur, Z1_s, Z1_uf, Z1_ur, Theta, ThetaVel, Z_rf, Z_rr in zip(Time, Zs, Zuf, Zur, Z1s, Z1uf, Z1ur, Pitch, Pitch_Vel, Zrf, Zrr):
       writer.writerow([t, Z_s, Z_uf, Z_ur, Z1_s, Z1_uf, Z1_ur, Theta, ThetaVel, Z_rf, Z_rr])

print(f'Saved {filename}')